In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("FinalData.csv", sep=";").fillna(0)
df = df[df["non_contextual_dist_E5"] != 0]
df.rename(columns={"EOS_E5": "CLS_E5"}, inplace=True)

In [ ]:
# Group data by bins
n_bins = 5  
data_series = pd.Series(df["lex_overlap"])
df["bin"] = pd.qcut(data_series, q=n_bins, duplicates='drop')
splits = [group.reset_index(drop=True) for _, group in df.groupby('bin')]

In [ ]:
from sklearn.metrics import roc_auc_score

# We calculate the auc dictionary for GPT first
GPT_dict = {}

for method in ["non_contextual_dist_GPT", "contextual_dist_GPT", "Centroid_GPT", "EOS_GPT", "VLDDR_GPT"]:
    GPT_dict[method] = {}

    # For each bin
    for dataframe in splits:
        auc = roc_auc_score(dataframe["same"], (-dataframe[method] if (method == "non_contextual_dist_GPT" or method == "contextual_dist_GPT") else dataframe[method]))
        GPT_dict[method][str(dataframe["bin"][0])] = auc

    # And generally
    auc = roc_auc_score(df["same"], (-df[method] if (method == "non_contextual_dist_GPT" or method == "contextual_dist_GPT") else df[method]))
    GPT_dict[method]["Overall"] = auc

GPT_dict["lex_overlap"] = {}
for dataframe in splits:
        auc = roc_auc_score(dataframe["same"], dataframe["lex_overlap"])
        GPT_dict["lex_overlap"][str(dataframe["bin"][0])] = auc
auc = roc_auc_score(df["same"], df["lex_overlap"])
GPT_dict["lex_overlap"]["Overall"] = auc   



# And then for E5
E5_dict = {}

for method in ["non_contextual_dist_E5", "contextual_dist_E5", "Centroid_E5", "CLS_E5", "VLDDR_E5"]:
    E5_dict[method] = {}

    # For each bin
    for dataframe in splits:
        auc = roc_auc_score(dataframe["same"], (-dataframe[method] if (method == "non_contextual_dist_E5" or method == "contextual_dist_E5") else dataframe[method]))
        E5_dict[method][str(dataframe["bin"][0])] = auc

    # And generally
    auc = roc_auc_score(df["same"], (-df[method] if (method == "non_contextual_dist_E5" or method == "contextual_dist_E5") else df[method]))
    E5_dict[method]["Overall"] = auc

In [ ]:
# Change dataframe keys
method_translation_list_GPT = [("lex_overlap", "Lexical Overlap"), ("non_contextual_dist_GPT", "Raw Embeddings Distance"), ("Centroid_GPT", "Centroid"), ("EOS_GPT", "EOS"), ("VLDDR_GPT", "VLDDR")]
method_translation_list_E5 = [("non_contextual_dist_E5", "Raw Embeddings Distance"), ("Centroid_E5", "Centroid"), ("CLS_E5", "CLS"), ("VLDDR_E5", "VLDDR")]

for translation in method_translation_list_GPT:
    GPT_dict[translation[1]] = GPT_dict.pop(translation[0])

GPT_dict.pop("contextual_dist_GPT")

for translation in method_translation_list_E5:
    E5_dict[translation[1]] = E5_dict.pop(translation[0])

E5_dict.pop("contextual_dist_E5")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

E5_df = pd.DataFrame(E5_dict)

#Split the data
df_main = E5_df.drop("Overall")
df_overall = E5_df.loc[["Overall"]]

vmin = E5_df.min().min()
vmax = E5_df.max().max()

#setup figure
fig = plt.figure(figsize=(8, 6))
gs = fig.add_gridspec(
    nrows=2, 
    ncols=2, 
    height_ratios=[len(df_main), 1], 
    width_ratios=[15, 1], 
    hspace=0.1,       
    wspace=0.05   
)

# setup subplots
ax_main = fig.add_subplot(gs[0, 0])
ax_overall = fig.add_subplot(gs[1, 0])
cbar_ax = fig.add_subplot(gs[:, 1]) 

# Plot top heatmap
sns.heatmap(
    df_main, 
    annot=True, 
    ax=ax_main, 
    cbar=True, 
    cbar_ax=cbar_ax, 
    vmin=vmin, 
    vmax=vmax,
    fmt=".4f"
)

# Plot bottom heatmap
sns.heatmap(
    df_overall, 
    annot=True, 
    ax=ax_overall, 
    cbar=False,     
    vmin=vmin, 
    vmax=vmax,
    fmt=".4f"
)

ax_main.set_title("AUC-ROC Scores E5 by Measure")
ax_main.set_xlabel("")            
ax_main.set_xticklabels([])    
ax_main.set_ylabel("Lexical Overlap Interval", fontweight = "bold")

ax_overall.set_xlabel("Measure", fontweight="bold")
ax_overall.set_ylabel("")        

for ax in [ax_main, ax_overall]:
    for _, spine in ax.spines.items():
        spine.set_visible(True)
        spine.set_color('black')
        spine.set_linewidth(1)

plt.show()

In [ ]:
GPT_df = pd.DataFrame(GPT_dict)
GPT_df_T = GPT_df.T
fig, ax = plt.subplots()
fig.patch.set_facecolor("white")
ax.set_facecolor("white")

GPT_df_T["Overall"].plot(
    kind="bar",
    ax=ax
)

ax.yaxis.grid(True, linestyle="--", linewidth=0.7, color="gray", alpha=0.7)
ax.set_axisbelow(True)

ax.set_xticklabels(
    [str(label).replace(" ", "\n") for label in GPT_df_T["Overall"].index],
    rotation=0
)

# Add white value labels inside each bar
for bar in ax.patches:
    ax.text(
        bar.get_x() + bar.get_width() / 2,  # Horizontal center of bar
        bar.get_height() / 2,               # Vertical center of bar
        f"{bar.get_height():.2f}",          # Value rounded to 2 decimals
        ha="center", va="center",
        color="white", fontsize=9, fontweight="bold"
    )

plt.ylabel("AUC-ROC scores", fontweight="bold")
plt.title("AUC-ROC values for GPT-2 by Measure")
plt.tight_layout()
plt.show()

In [ ]:
#Calculations for statistical significance
from MLstatkit.stats import Delong_test

print("GPT-2")
for method in ["lex_overlap", "non_contextual_dist_GPT", "Centroid_GPT", "EOS_GPT"]:
    z, p_value = Delong_test(df["same"], df["VLDDR_GPT"], df[method])
    print(" " + method)
    print(f"  p: {p_value:.4e}")

print("E5")
for method in ["lex_overlap", "non_contextual_dist_E5", "Centroid_E5", "CLS_E5"]:
    z, p_value = Delong_test(df["same"], df["VLDDR_E5"], df[method])
    print(" " + method)
    print(f"  p-value: {p_value:.4e}")